# Fused Geospatial Embeddings from the KTH team

## 1. Clone the Coperinicus-FM dependency

In [1]:
!git clone https://github.com/zhu-xlab/Copernicus-FM.git foundation_embeddings/Copernicus-FM

Cloning into './embed2scale-solution/foundation_embeddings/Copernicus-FM'...
remote: Enumerating objects: 531, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 531 (delta 62), reused 73 (delta 27), pack-reused 396 (from 1)
Receiving objects: 100% (531/531), 30.46 MiB | 4.03 MiB/s, done.
Resolving deltas: 100% (243/243), done.


## 2. Install dependencies

In [2]:
!pip install -q timm einops rasterio pyproj torchgeo xarray zarr wandb

## 3. Download the pretrained foundation models

We save them under `foundation_embeddings/pretrained_models/`.

In [3]:
!mkdir -p foundation_embeddings/pretrained_models
!wget -c https://huggingface.co/wangyi111/Copernicus-FM/resolve/main/CopernicusFM_ViT_base_varlang_e100.pth -P foundation_embeddings/pretrained_models/
!wget -c https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-600M/resolve/main/Prithvi_EO_V2_600M.pt -P foundation_embeddings/pretrained_models/
!wget -c https://github.com/bair-climate-initiative/scale-mae/releases/download/base-800/scalemae-vitlarge-800.pth -P foundation_embeddings/pretrained_models/
!wget -c https://huggingface.co/antofuller/CROMA/resolve/main/CROMA_large.pt -P foundation_embeddings/pretrained_models/
!ls -lh foundation_embeddings/pretrained_models

--2026-08-08 22:07:11--  https://huggingface.co/wangyi111/Copernicus-FM/resolve/main/CopernicusFM_ViT_base_varlang_e100.pth
Resolving huggingface.co (huggingface.co)... 2600:9000:26a1:0:17:b174:6d00:93a1, 2600:9000:26a1:3e00:17:b174:6d00:93a1, 2600:9000:26a1:fa00:17:b174:6d00:93a1, ...
Connecting to huggingface.co (huggingface.co)|2600:9000:26a1:0:17:b174:6d00:93a1|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/679a8c54bc34f5d20e88e5a1/50e73a8e5867a75a0bb63d5d5ef4af4d3f6bf8f25d8df98f54588e3295bf1d52?user_id=public&X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27CopernicusFM_ViT_base_varlang_e100.pth%3B+filename%3D%22CopernicusFM_ViT_base_varlang_e100.pth%22%3B&Expires=1786223232&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjc5YThjNTRiYzM0ZjVkMjBlODhlNWExLzUwZTczYThlNTg2N2E3NWEwYmI2M2Q1ZDVlZjRhZjRkM2Y2YmY4ZjI1ZDhkZjk4ZjU0NTg4ZTMyOTV

## 4. Generate foundation-model embeddings

Let's use our five foundation models to generate embeddings of our training data.

In [ ]:
embedding_data_dir = "../data"
downstream_data_dir = "../../../data/data_example"

%cd foundation_embeddings

In [ ]:
!python3 embed_copernicus.py {downstream_data_dir} {embedding_data_dir}/copernicus-fm/copernicusfm_concat_temporal_raw_4x768.npz

!python3 embed_dofa.py {downstream_data_dir} {embedding_data_dir}/dofa/dofa_temporal_raw_4x1536.npz

!python3 embed_croma.py {downstream_data_dir} {embedding_data_dir}/croma-embeddings/embeddings

!python3 embed_prithvi.py {downstream_data_dir} {embedding_data_dir}/prithvi2/prithvi2_1x1536.npz

!python3 embed_scale_mae.py {downstream_data_dir} {embedding_data_dir}/scalemae/scalemae_temporal_data_4x1536.npz

In [ ]:
%cd ../

## 5. Train the autoencoder

Train an autoencoder on the extracted foundation-model embeddings, to compact their size to what's required for Neuco-Bench.

### Imports

In [1]:
import os
import wandb
wandb.init(mode="disabled")

import pathlib
import random
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from train_encoder import fix_seeds, train, extract, collate_fn
from decoder import LinearDecoder
from embedding_datasets import (
    EmbeddingDataset,
    MergedEmbeddingDataset,
    SingleFileEmbeddingDataset,
)

### Set up training parameters

In [20]:
# Data params
seed = 1337
batch_size = 32
val_batch_size = 8
num_workers = 8
val_num_workers = 8
data_directory = pathlib.Path("./data/")
experiment_dir_prefix = "experiments"

# Training params
lr = 1e-4
loss_weights = {
    "copernicus": 1.0,
    "dofa": 1.0,
    "croma": 0.5,
    "prithvi": 2.0,
    "scalemae": 0.75,
}
n_epochs = 160
lr_milestones = [0.6, 0.9]

# Some bookkeeping
experiment_name = (
    "linear_embed"
    + f"_{lr}_{n_epochs}_"
    + time.strftime("%Y%m%d_%H%M%S", time.localtime())
)
experiment_directory = pathlib.Path(experiment_dir_prefix) / experiment_name
experiment_directory.mkdir(exist_ok=True)

fix_seeds(seed)

if torch.cuda.is_available():
    device = torch.device("cuda", 0)
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")
    torch.cpu.set_device(device)

### Define encoder/decoder

Define a simple encoder/decoder architecture with a single GELU nonlinearity and two dropout layers. 

In [3]:
from einops import rearrange

class FFNEncoder(nn.Module):
    def __init__(
        self,
        input_size: list[list[int]],
        output_dim: int = 1024,
        hidden_dim: int = 2048,
    ) -> None:
        super().__init__()

        self.input_dim = np.sum([np.prod(il) for il in input_size])
        self.output_dim = output_dim

        self.activation_function = nn.GELU()
        self.dropout = nn.Dropout(p=0.1)
        self.input_dropout = nn.Dropout(p=0.1)
        self.input_norm = nn.LayerNorm(self.input_dim)

        self.net = nn.Sequential(
            self.input_norm,
            self.input_dropout,
            nn.Linear(self.input_dim, hidden_dim),
            self.activation_function,
            self.dropout,
            nn.Linear(hidden_dim, self.output_dim),
        )

    def forward(self, x: dict[str, torch.Tensor]) -> list[torch.Tensor]:
        x = torch.cat([rearrange(t, "b c t -> b (c t)") for t in x.values()], dim=1)

        return self.net(x)

The decoder runs it's input through the encoder, then tires to recover the input using a single linear layer.

In [4]:
class LinearDecoder(nn.Module):
    def __init__(
        self,
        encoder: FFNEncoder,
    ):
        super().__init__()
        self.encoder = encoder
        self.net = nn.Linear(encoder.output_dim, encoder.input_dim)

    def forward(self, x: dict[str, torch.Tensor]) -> torch.Tensor:
        input_shapes = {k: t.shape for k, t in x.items()}
        feat = self.encoder(x)
        logits = self.net(feat)
        logits = logits.split([np.prod(s[1:]) for s in input_shapes.values()], dim=1)

        logits = {
            k: logits[i].reshape(input_shapes[k])
            for i, k in enumerate(input_shapes.keys())
        }
        return logits

In [ ]:
encoder = FFNEncoder(
    # These are the sizes of the embeddings for each of the foundation networks
    # [Copernicus, Dofa, Croma, Prithvi, Scale-MAE]
    input_size=[[4, 768], [4, 1536], [4, 1024], [1, 1536], [4, 1536]],
)
decoder = LinearDecoder(encoder)
decoder.to(device)

### Load the embeddings generated by the doundation models

In [5]:
dataset = MergedEmbeddingDataset(
    datasets=[
        SingleFileEmbeddingDataset(
            path=data_directory
            / "copernicus-fm/copernicusfm_concat_temporal_raw_4x768.npz",
            dataset_key="copernicus",
        ),
        SingleFileEmbeddingDataset(
            path=data_directory / "dofa/dofa_temporal_raw_4x1536.npz",
            dataset_key="dofa",
        ),
        EmbeddingDataset(
            root_paths={"croma": data_directory / "croma-embeddings"}
        ),
        SingleFileEmbeddingDataset(
            path=data_directory / "prithvi2/prithvi2_1x1536.npz",
            dataset_key="prithvi",
        ),
        SingleFileEmbeddingDataset(
            path=data_directory / "scalemae/scalemae_temporal_data_4x1536.npz",
            dataset_key="scalemae",
        ),
    ]
)

print(f"Length of training dataset: {len(dataset)}")

data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=False,
    drop_last=True,
    shuffle=True,
    collate_fn=collate_fn,
)

Length of training dataset: 50


### Run the training loop

Note: This is a bog standard pytorch training loop with a single mean square error reconstruction loss. It's defined in `train_encoder.py`, if you want to check more details.

In [19]:
optimizer = torch.optim.AdamW(
    params=decoder.parameters(),
    lr=lr,
    weight_decay=0.05,
)

lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer,
    [int(len(data_loader) * n_epochs * r) for r in lr_milestones],
    gamma=0.1,
)

train(
    model=decoder,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    exp_dir=experiment_directory,
    epochs=n_epochs,
    dataloader=data_loader,
    val_loader=data_loader,
    loss_weights=loss_weights,
    device=device,
)

Training:   0%|                                                                                                                                                                      | 0/160 [00:00<?, ?it/s]

Evaluation losses: 	scalemae:   0.0711  prithvi:    0.014  croma:   0.0627  dofa:   0.0596  copernicus:   0.0413  MSE:    0.213


Training:   0%|                                                                                                                                                                      | 0/160 [00:00<?, ?it/s]

Epoch 0 | Checkpoint saved at experiments/linear_embed_0.0001_160_20260809_100536/checkpoint_best.pth


Training:   0%|                                                                                                                                                                      | 0/160 [00:01<?, ?it/s]

Epoch 0 | Checkpoint saved at experiments/linear_embed_0.0001_160_20260809_100536/checkpoint_0.pth


Training:   1%|▉                                                                                                                                                             | 1/160 [00:02<05:29,  2.07s/it]

Epoch 0 | Training losses: 	copernicus:   0.0511  dofa:   0.0643  croma:   0.0366  prithvi:   0.0558  scalemae:   0.0635  MSE:    0.271


Training:   1%|▉                                                                                                                                                             | 1/160 [00:02<06:02,  2.28s/it]


KeyboardInterrupt: 

## 6. Generating compressed embeddings for NeuCo Bench 
We run our dataset through the encoder, and package the results into a NeuCo-Bench compatible csv file.

In [ ]:
extract(encoder, "checkpoint_best.pth", experiment_directory, dataset, device)

!python3 embeddings_to_csv.py {experiment_directory} submission.csv

Let's check the exported csv. From here you can run NeuCo-Bench on these embeddings just like in the previous tutorial.

In [17]:
import pandas as pd

submission = pd.read_csv("submission.csv")

print(submission)

                                                   id      0       1      2  \
0   0002c8e787ba871d725f57833996ef31a4d60f370180a9...  3.479  1.5390 -1.976   
1   00538cb13e3071d4fd3d747a12ee4817d05f437e20849a...  1.622  0.9640 -1.814   
2   01b56662447728fb2610e469b8412f46debab42d9c6e7c...  3.219  1.7190 -2.014   
3   0219ab6c2d6275854d5a6506b0062c23e81ddf65e8d471...  3.117  1.4690 -2.057   
4   0247b254eb32f76d8145f33d21a8efe4475ce1f4f9abc7...  3.299  1.8180 -1.799   
5   0287b0e93b630e1a14db6b7c085ea138d4e97b3f7e8e81...  2.014  1.0350 -1.733   
6   02a94579889994622e2dfa67dd05ef10529d91f6007dac...  2.797  0.7656 -2.123   
7   02c0791bb9ea4da49b594764470001e89319cf6e0fd780...  2.780  0.9870 -2.680   
8   02cc10d8ece27794348b8cfc713c0d51e3f04338f5bceb...  3.295  1.7190 -1.944   
9   02dd43a8a486f932f19a6701c9f5dd3122752d89fc0ca3...  3.234  1.6045 -2.115   
10  02f05c5b5883117654f0f6fd5ef4ed47e0eafd01d3cc9d...  2.310  0.6850 -1.865   
11  02f2fa6750c982f03337dc7af557e5cc207b16a4620265..